In [1]:
import scanpy as sc
import anndata as ad
import scib
import matplotlib.pyplot as plt
import pandas as pd

/rds/general/user/ztb25/home/miniforge3/envs/integrate/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [ ]:
# load in dataset
adata = sc.read("/rds/general/user/ztb25/home/PBMC_datasets/integrated_adata.h5ad")

In [ ]:
## Benchmark with coarse cell-type labels
batch_key = "dataset" 
label_key = "celltypist_cell_label_coarse"

# need to recompute neighbours for each embedding (for graph_conn metric), consider redoing integration with new adatas per method to make this cleaner
def metrics_with_neighbors(adata, batch_key, label_key, embed):
    """Recompute default neighbours from `embed` so graph_conn reflects the method."""
    sc.pp.neighbors(adata, use_rep=embed)   # writes to default slot
    return scib.metrics.metrics_fast(adata, adata, batch_key, label_key, embed=embed)

metrics_scvi      = metrics_with_neighbors(adata, batch_key, label_key, "X_scVI")
metrics_scanvi    = metrics_with_neighbors(adata, batch_key, label_key, "X_scANVI")
metrics_harmony   = metrics_with_neighbors(adata, batch_key, label_key, "X_pca_harmony")
metrics_scanorama = metrics_with_neighbors(adata, batch_key, label_key, "X_scanorama")
metrics_unintegrated = metrics_with_neighbors(adata, batch_key, label_key, "X_pca")

# BBKNN: restore the saved BBKNN graph to default slot, then run without embed=
adata.uns["neighbors"]       = adata.uns["bbknn"].copy()
adata.obsp["distances"]      = adata.obsp["distances_bbknn"].copy()
adata.obsp["connectivities"] = adata.obsp["connectivities_bbknn"].copy()
metrics_bbknn = scib.metrics.metrics_fast(adata, adata, batch_key, label_key)

In [ ]:
# plot all metrics together for comparison

# Concatenate metrics results
metrics = pd.concat(
    [metrics_scvi, metrics_scanvi, metrics_harmony, metrics_scanorama, metrics_bbknn, metrics_unintegrated],
    axis="columns",
)
# Set methods as column names
metrics = metrics.set_axis(
    ["scVI", "scANVI", "Harmony", "Scanorama", "BBKNN", "Unintegrated"], axis="columns"
)
# Select only the fast metrics
metrics = metrics.loc[
    [
        "ASW_label",
        "ASW_label/batch",
        "PCR_batch",
        "isolated_label_silhouette",
        "graph_conn",
        "hvg_overlap",
    ],
    :,
]
# Transpose so that metrics are columns and methods are rows
metrics = metrics.T
# Remove the HVG overlap metric because it's not relevant to embedding outputs
metrics = metrics.drop(columns=["hvg_overlap"])
metrics

In [ ]:
metrics.style.background_gradient(cmap="Blues")

In [ ]:
# Scale 0–1 across methods to emphasise differences
metrics_scaled = (metrics - metrics.min()) / (metrics.max() - metrics.min())
metrics_scaled.style.background_gradient(cmap="Blues")

In [ ]:
# Group into Batch correction vs Biological conservation summaries
metrics_scaled["Batch"] = metrics_scaled[
    ["ASW_label/batch", "PCR_batch", "graph_conn"]
].mean(axis=1)
metrics_scaled["Bio"] = metrics_scaled[
    ["ASW_label", "isolated_label_silhouette"]
].mean(axis=1)
metrics_scaled.style.background_gradient(cmap="Blues")

In [ ]:
# Scatter: Batch vs Bio trade-off
fig, ax = plt.subplots()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
metrics_scaled.plot.scatter(
    x="Batch",
    y="Bio",
    c=range(len(metrics_scaled)),
    ax=ax,
)
for k, v in metrics_scaled[["Batch", "Bio"]].iterrows():
    ax.annotate(
        k, v,
        xytext=(6, -3), textcoords="offset points",
        family="sans-serif", fontsize=12,
    )
plt.show()

In [ ]:
# Weighted overall score (40 % batch, 60 % bio per scIB paper)
metrics_scaled["Overall"] = (
    0.4 * metrics_scaled["Batch"] + 0.6 * metrics_scaled["Bio"]
)
metrics_scaled.style.background_gradient(cmap="Blues")

In [ ]:
# Bar chart of overall ranking
metrics_scaled.plot.bar(y="Overall")
plt.show()

In [ ]:
# ## Benchmark with fine cell-type labels
# label_key = "celltypist_cell_label_fine"

# metrics_scvi      = metrics_with_neighbors(adata, batch_key, label_key, "X_scVI")
# metrics_scanvi    = metrics_with_neighbors(adata, batch_key, label_key, "X_scANVI")
# metrics_harmony   = metrics_with_neighbors(adata, batch_key, label_key, "X_pca_harmony")
# metrics_scanorama = metrics_with_neighbors(adata, batch_key, label_key, "X_scanorama")
# metrics_unintegrated = metrics_with_neighbors(adata, batch_key, label_key, "X_pca")

# # BBKNN: restore the saved BBKNN graph to default slot, then run without embed=
# adata.uns["neighbors"]       = adata.uns["bbknn"].copy()
# adata.obsp["distances"]      = adata.obsp["distances_bbknn"].copy()
# adata.obsp["connectivities"] = adata.obsp["connectivities_bbknn"].copy()
# metrics_bbknn = scib.metrics.metrics_fast(adata, adata, batch_key, label_key)


In [ ]:
# # plot all metrics together for comparison

# # Concatenate metrics results
# metrics = pd.concat(
#     [metrics_scvi, metrics_scanvi, metrics_harmony, metrics_scanorama, metrics_bbknn, metrics_unintegrated],
#     axis="columns",
# )
# # Set methods as column names
# metrics = metrics.set_axis(
#     ["scVI", "scANVI", "Harmony", "Scanorama", "BBKNN", "Unintegrated"], axis="columns"
# )
# # Select only the fast metrics
# metrics = metrics.loc[
#     [
#         "ASW_label",
#         "ASW_label/batch",
#         "PCR_batch",
#         "isolated_label_silhouette",
#         "graph_conn",
#         "hvg_overlap",
#     ],
#     :,
# ]
# # Transpose so that metrics are columns and methods are rows
# metrics = metrics.T
# # Remove the HVG overlap metric because it's not relevant to embedding outputs
# metrics = metrics.drop(columns=["hvg_overlap"])
# metrics

In [ ]:
# metrics.style.background_gradient(cmap="Blues")

In [ ]:
# # Scale 0–1 across methods to emphasise differences
# metrics_scaled = (metrics - metrics.min()) / (metrics.max() - metrics.min())
# metrics_scaled.style.background_gradient(cmap="Blues")

In [ ]:
# # Group into Batch correction vs Biological conservation summaries
# metrics_scaled["Batch"] = metrics_scaled[
#     ["ASW_label/batch", "PCR_batch", "graph_conn"]
# ].mean(axis=1)
# metrics_scaled["Bio"] = metrics_scaled[
#     ["ASW_label", "isolated_label_silhouette"]
# ].mean(axis=1)
# metrics_scaled.style.background_gradient(cmap="Blues")

In [ ]:
# # Scatter: Batch vs Bio trade-off
# fig, ax = plt.subplots()
# ax.set_xlim(0, 1)
# ax.set_ylim(0, 1)
# metrics_scaled.plot.scatter(
#     x="Batch",
#     y="Bio",
#     c=range(len(metrics_scaled)),
#     ax=ax,
# )
# for k, v in metrics_scaled[["Batch", "Bio"]].iterrows():
#     ax.annotate(
#         k, v,
#         xytext=(6, -3), textcoords="offset points",
#         family="sans-serif", fontsize=12,
#     )
# plt.show()

In [ ]:
# # Weighted overall score (40 % batch, 60 % bio per scIB paper)
# metrics_scaled["Overall"] = (
#     0.4 * metrics_scaled["Batch"] + 0.6 * metrics_scaled["Bio"]
# )
# metrics_scaled.style.background_gradient(cmap="Blues")

In [ ]:
# # Bar chart of overall ranking
# metrics_scaled.plot.bar(y="Overall")
# plt.show()